In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Coherence-Accuracy Correlation and Metrics Table

In [ ]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300

# Paths
RESULTS_DIR = Path("../results")
PLOTS_DIR = Path(".")
PLOTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Load data
df_70b = pd.read_csv(RESULTS_DIR / "eval_gpt-4o-mini_var_Llama-3.3-70B-Instruct_dt_20250929_1404_answers_with_subject.csv")
df_8b = pd.read_csv(RESULTS_DIR / "eval_gpt-4o-mini_var_Meta-Llama-3.1-8B-Instruct_dt_20250929_1201_answers_with_subject.csv")

# Add model identifier
df_70b['model'] = 'Llama-3.3-70B'
df_8b['model'] = 'Llama-3.1-8B'

# Combine datasets
df_combined = pd.concat([df_70b, df_8b], ignore_index=True)

# Create binary accuracy from result
df_combined['is_correct'] = df_combined['result'].astype(str).str.lower().str.startswith('hit')

# Convert coherence to numeric
df_combined['coherence_num'] = pd.to_numeric(df_combined['coherence'], errors='coerce')

# Convert behavior to numeric (assuming it's like coherence)
df_combined['behavior_num'] = pd.to_numeric(df_combined['behavior'], errors='coerce')

## Coherence-Accuracy Correlation Plot

In [ ]:
# Create coherence bins
df_plot = df_combined[['coherence_num', 'is_correct', 'model']].dropna()
df_plot['coherence_bin'] = pd.cut(df_plot['coherence_num'], bins=10, labels=False)

# Calculate mean accuracy per bin for each model
binned_stats = df_plot.groupby(['model', 'coherence_bin']).agg({
    'is_correct': 'mean',
    'coherence_num': 'mean'
}).reset_index()

# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

# Calculate correlations first
corr_70b = df_plot[df_plot['model'] == 'Llama-3.3-70B'][['coherence_num', 'is_correct']].corr().iloc[0, 1]
corr_8b = df_plot[df_plot['model'] == 'Llama-3.1-8B'][['coherence_num', 'is_correct']].corr().iloc[0, 1]

for model in ['Llama-3.3-70B', 'Llama-3.1-8B']:
    model_data = binned_stats[binned_stats['model'] == model]
    corr = corr_70b if model == 'Llama-3.3-70B' else corr_8b
    ax.scatter(model_data['coherence_num'], model_data['is_correct'],
               label=f'{model} (r={corr:.3f})', s=150, alpha=0.7)
    # Add trend line
    z = np.polyfit(model_data['coherence_num'], model_data['is_correct'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(model_data['coherence_num'].min(),
                          model_data['coherence_num'].max(), 100)
    ax.plot(x_line, p(x_line), "--", alpha=0.5, linewidth=2)

ax.set_xlabel('Mean Coherence Score (binned)', fontsize=14, fontweight='bold')
ax.set_ylabel('Mean Accuracy', fontsize=14, fontweight='bold')
ax.set_title('Correlation between Coherence and Accuracy\n(Averaged over coherence bins)',
             fontsize=18, fontweight='bold')
legend = ax.legend(title='Pearson r (instance-level)', loc='lower right', fontsize=12, title_fontsize=14)
legend.get_title().set_fontweight('bold')
ax.grid(True, alpha=0.3)
ax.tick_params(axis='both', labelsize=12)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "coherence_accuracy_correlation.jpg", dpi=300, bbox_inches='tight')
plt.savefig(PLOTS_DIR / "coherence_accuracy_correlation.png", dpi=300, bbox_inches='tight')
plt.savefig(PLOTS_DIR / "coherence_accuracy_correlation.pdf", dpi=300, bbox_inches='tight', format='pdf')
print(f"✓ Saved correlation plot to {PLOTS_DIR / 'coherence_accuracy_correlation.jpg'} and .pdf")
plt.show()

## Metrics Summary Table

In [ ]:
# Calculate mean metrics per steering method and model
metrics_table = df_combined.groupby(['model', 'steering_method']).agg({
    'is_correct': 'mean',
    'behavior_num': 'mean',
    'coherence_num': 'mean'
}).reset_index()

# Rename columns
metrics_table.columns = ['Model', 'Steering Method', 'Accuracy', 'Behavior', 'Coherence']

# Round to 3 decimal places
metrics_table[['Accuracy', 'Behavior', 'Coherence']] = metrics_table[['Accuracy', 'Behavior', 'Coherence']].round(3)

# Save to CSV
metrics_table.to_csv(RESULTS_DIR / "metrics_by_method_and_model.csv", index=False)
print(f"\n✓ Saved metrics table to {RESULTS_DIR / 'metrics_by_method_and_model.csv'}")

# Print the table
print("\n" + "="*80)
print("METRICS BY STEERING METHOD AND MODEL")
print("="*80)
print(metrics_table.to_string(index=False))
print("="*80)

## Pivot Tables

In [ ]:
print("\n" + "="*80)
print("PIVOT TABLE: ACCURACY")
print("="*80)
accuracy_pivot = metrics_table.pivot(index='Steering Method', columns='Model', values='Accuracy')
print(accuracy_pivot.to_string())

print("\n" + "="*80)
print("PIVOT TABLE: BEHAVIOR")
print("="*80)
behavior_pivot = metrics_table.pivot(index='Steering Method', columns='Model', values='Behavior')
print(behavior_pivot.to_string())

print("\n" + "="*80)
print("PIVOT TABLE: COHERENCE")
print("="*80)
coherence_pivot = metrics_table.pivot(index='Steering Method', columns='Model', values='Coherence')
print(coherence_pivot.to_string())
print("="*80)